In [1]:
import os
import cv2 as cv
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


In [22]:
# Set the path to your training images
path = r"C:\Users\RAHUL PATIL\Downloads\archive (17)\Train"
categories = os.listdir(path)
img_size = 224

data = []
labels = []

for idx, category in enumerate(categories):
    folder_path = os.path.join(path, category)
    for img_file in os.listdir(folder_path):
        try:
            img_path = os.path.join(folder_path, img_file)
            img = cv.imread(img_path)
            img = cv.resize(img, (img_size, img_size))
            data.append(img)
            labels.append(idx)
        except:
            pass

X = np.array(data) / 255.0  
y = to_categorical(labels)


In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [24]:
base_model = MobileNetV2(include_top=False, input_shape=(img_size, img_size, 3), weights='imagenet')
base_model.trainable = False  # Freeze pre-trained layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(len(categories), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [25]:
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("currency_note_model.keras", save_best_only=True)
]

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=30,
    callbacks=callbacks
)


Epoch 1/30


C:\Users\RAHUL PATIL\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.0917 - loss: 2.6315 - val_accuracy: 0.1042 - val_loss: 1.8676
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 760ms/step - accuracy: 0.1930 - loss: 2.0922 - val_accuracy: 0.2708 - val_loss: 1.6723
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 767ms/step - accuracy: 0.3871 - loss: 1.6678 - val_accuracy: 0.3333 - val_loss: 1.5579
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 757ms/step - accuracy: 0.3996 - loss: 1.6412 - val_accuracy: 0.3958 - val_loss: 1.4796
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 715ms/step - accuracy: 0.4078 - loss: 1.4765 - val_accuracy: 0.4375 - val_loss: 1.3714
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 734ms/step - accuracy: 0.4579 - loss: 1.3855 - val_accuracy: 0.4375 - val_loss: 1.3059
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 743ms/step - accuracy: 0.5202 - loss: 1.2970 - val_accuracy: 0.5000 - val_loss: 1.2769
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 718ms/step - accuracy: 0.5442 - loss: 1.2942 - val_accuracy: 0.5625 - val_loss: 1.2003
Epo

In [26]:
model.save("currency_note_model.keras")


In [27]:
def predict_note(img_path):
    model = load_model("currency_note_model.keras")
    categories = ['1Hundrednote', '2Hundrednote', '2Thousandnote', '5Hundrednote', 'Fiftynote', 'Tennote', 'Twentynote']
    img_size = 224

    img = cv.imread(img_path)
    img = cv.resize(img, (img_size, img_size))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction)

    print(f"Predicted: {categories[class_index]} ({confidence*100:.2f}% confidence)")


In [38]:
import cv2

cap = cv2.VideoCapture(0)  

if not cap.isOpened():
    print("❌ Could not open webcam.")
else:
    print("✅ Webcam is working. Press SPACE to capture image, ESC to exit.")
    captured_image = None  # Initialize a variable to store the captured image

    while True:
        ret, frame = cap.read()
        if not ret:
            print("⚠️ Failed to grab frame.")
            break

        cv2.imshow("Webcam Test", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:  # ESC key to exit
            break
        elif key == 32:  # SPACE key to capture image
            captured_image = frame.copy() # Store a copy of the frame
            print("📸 Image captured! Stored in 'captured_image' variable.")
            break
cap.release()
cv2.destroyAllWindows()

# After the loop, you can access the captured image from the 'captured_image' variable
if captured_image is not None:
    cv2.imwrite("captured_image.jpg", captured_image)
    print("Image saved as captured_image.jpg")
else:
    print("No image was captured.")

✅ Webcam is working. Press SPACE to capture image, ESC to exit.
📸 Image captured! Stored in 'captured_image' variable.
Image saved as captured_image.jpg


In [39]:
predict_note(r"captured_image.jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted: Tennote (35.66% confidence)
